In [57]:
!pip install skyfield
!pip install tensorflow

In [58]:
from skyfield.api import EarthSatellite, load, utc
from datetime import datetime, timedelta
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,LSTM
from sklearn.metrics import mean_squared_error
import plotly.graph_objects as go

ts = load.timescale()

In [59]:
x = []
y = []
z = []
times = []

In [60]:
satellite = """IRIS
            1 39197U 13033A   25231.57687540  .00001065  00000+0  13165-3 0  9994
            2 39197  97.9505  66.6828 0023900 222.8047 137.1307 14.84728440655629"""

tle_lines = [line.strip() for line in satellite.strip().split('\n')]

end_time = datetime.now(tz = utc)    ### 15 days with 5 min gap
start_time = end_time - timedelta(days=15)
delta = timedelta(minutes=5)

sat_info = EarthSatellite(tle_lines[1], tle_lines[2], tle_lines[0], ts)   #EarthSattilite object

while start_time <= end_time :  #list of time
  times.append(ts.utc(start_time))
  start_time += delta



for time in times :
  geo = sat_info.at(time)
  pos = geo.position.km
  x.append(pos[0])
  y.append(pos[1])
  z.append(pos[2])

x = np.array(x).reshape(-1,1)
y = np.array(y).reshape(-1,1)
z = np.array(z).reshape(-1,1)

print(x.shape)

cords = np.hstack((x,y,z))
cords.shape

(4321, 1)


(4321, 3)

In [61]:
features = []
next_pos = []

for i in range(x.shape[0] - 10):
  features.append(cords[i : i+10])
  next_pos.append(cords[i+10])

features = np.array(features)
next_pos = np.array(next_pos)

print(features.shape)
print(next_pos.shape)

(4311, 10, 3)
(4311, 3)


In [62]:
f_train, f_test, n_train, n_test = train_test_split(features, next_pos, test_size=0.2, random_state=33)
print(f_train.shape)
print(n_train.shape)
print(f_test.shape)
print(n_test.shape)

(3448, 10, 3)
(3448, 3)
(863, 10, 3)
(863, 3)


In [63]:
model = Sequential()
model.add(LSTM(256,input_shape=(10,3)))
model.add(Dense(128))
model.add(Dense(64))
model.add(Dense(32))
model.add(Dense(3))
model.compile(loss='mean_squared_error',optimizer='adam')
model.fit(f_train, n_train, epochs=70)
predicted = model.predict(f_test)
print('actual',n_test)
print('predicted',predicted)

Epoch 1/70


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



108/108 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - loss: 14417374.0000
Epoch 2/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 96703.2109
Epoch 3/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - loss: 50982.6953
Epoch 4/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 50600.9805
Epoch 5/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - loss: 54234.5664
Epoch 6/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 53110.7422
Epoch 7/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 54008.2422
Epoch 8/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 53945.2969
Epoch 9/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 51938.4570
Epoch 10/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 53088.8008
Epoch 11/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 52829.0547
Epoch 12/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - loss: 53083.0898
Epoch 13/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 54499.5547
Epoch 14/70
108/108 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 52887.

In [64]:
rmse = np.sqrt(mean_squared_error(n_test, predicted))
print('RMSE:', rmse)

RMSE: 233.3234027860233


In [65]:

error = np.abs(n_test - predicted)
print(error)

distanceError = []

for i in range(len(error)):
    dist = np.sqrt(error[i][0]**2 + error[i][1]**2 + error[i][2]**2)
    distanceError.append(dist)

print("Error",distanceError)

distanceError = np.array(distanceError)

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(len(distanceError)), y=distanceError, name='Distance Error (km)', marker_color='indianred'))
fig.update_layout(
    title='Distance Error between Actual and Predicted Positions using Random Forest Model',
    xaxis_title='Test Sample Index',
    yaxis_title='Distance Error (km)',
)
fig.show()

[[ 25.52504847  35.25596629  25.75550351]
 [427.39570972  59.11382451 187.01340704]
 [362.76283188 138.4123144    1.70376659]
 ...
 [ 24.49018611   7.34526623 119.69908744]
 [206.08656324 131.42163155  97.89829568]
 [116.86756477  74.94832823  56.60704649]]
Error [np.float64(50.57526291780698), np.float64(470.2505197702575), np.float64(388.27534533250844), np.float64(436.85699217944745), np.float64(61.49641056134723), np.float64(401.87547140412113), np.float64(316.18510719408755), np.float64(857.6637387648284), np.float64(141.1351530901799), np.float64(374.15245976233666), np.float64(325.215225877092), np.float64(400.1726731031584), np.float64(301.06181729012644), np.float64(311.1245000808399), np.float64(239.57395520196164), np.float64(95.23722738878561), np.float64(156.6940550946648), np.float64(429.5194924006547), np.float64(136.0236440888605), np.float64(413.45465128415043), np.float64(259.002288420358), np.float64(214.64440329801968), np.float64(778.9329336315451), np.float64(205.